# 🌿 Plant Disease Identification using Machine Learning
---
| Step | Task | Member |
|------|------|--------|
| 1 | Data Cleaning | Tanusha |
| 2 | Data Validation | Zaina |
| 3 | Data Augmentation | Vishwa |
| 4 | Feature Engineering | Rakesh |
| 5 | Train/Test Split | Siri |
| 6 | Feature Processing | Ranjith |
| 7 | Model Training | Vaseem |
| 8 | Model Evaluation | Rudresh |
| 9 | Model Architecture & Testing | Shankar |

> **Memory-safe version** — uses `ImageDataGenerator.flow_from_directory()` for CNN
> and a 5 000-sample draw for SVM/RF. No full dataset is ever loaded into RAM.


## 📦 Step 0 — Import Required Libraries

In [2]:
import os
import gc
import pickle
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                     Dense, Dropout, BatchNormalization)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("✅ All libraries imported successfully.")

ImportError: DLL load failed while importing _op_def_registry: An Application Control policy has blocked this file.

## 🧹 Step 1 — Data Cleaning  `[Tanusha]`
Remove corrupted or unreadable images before any processing.

In [ ]:
# ── UPDATE THIS PATH ─────────────────────────────────────────────────────────
dataset_path = r"D:\VS Code\Python\plant disease idetification\plantvillage dataset\color"
# ─────────────────────────────────────────────────────────────────────────────

removed = 0
for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)
    if not os.path.isdir(folder_path):
        continue
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            continue
        try:
            img = Image.open(fpath)
            img.verify()
        except Exception:
            print(f"Removing corrupted: {fpath}")
            os.remove(fpath)
            removed += 1

print(f"✅ Data Cleaning complete. Removed {removed} corrupted file(s).")

## ✅ Step 2 — Data Validation  `[Zaina]`
List all classes, count images per class, flag empty folders.

In [ ]:
classes = sorted([
    f for f in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, f))
])
print(f"Total classes found: {len(classes)}\n")

total_images = 0
empty_classes = []
for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    print(f"  {cls:<60} → {len(imgs):>5} images")
    total_images += len(imgs)
    if len(imgs) == 0:
        empty_classes.append(cls)

print(f"\nTotal images across all classes: {total_images}")
if empty_classes:
    print(f"⚠️  Empty classes: {empty_classes}")
else:
    print("✅ Data Validation complete. No empty classes found.")

## 🔄 Step 3 — Data Augmentation  `[Vishwa]`
Configure augmentation for training. Applied live during model.fit via the generator — no extra RAM needed.

In [ ]:
# Global size and batch constants used in all subsequent steps
IMAGE_SIZE = 32      # 32×32 → streams from disk with minimal RAM
BATCH_SIZE = 32

# Training generator: rescale + augmentation
datagen_train = ImageDataGenerator(
    rescale          = 1./255,
    validation_split = 0.2,       # 80/20 split handled inside the generator
    rotation_range   = 20,
    width_shift_range= 0.1,
    height_shift_range=0.1,
    horizontal_flip  = True,
    zoom_range       = 0.1,
    fill_mode        = 'nearest'
)

# Validation generator: rescale only (no augmentation on val data)
datagen_val = ImageDataGenerator(
    rescale          = 1./255,
    validation_split = 0.2
)

print("✅ Augmentation generators configured.")
print("   Training  : rescale + rotation + shift + flip + zoom")
print("   Validation: rescale only")

## 🛠️ Step 4 — Feature Engineering / Data Generators  `[Rakesh]`
`flow_from_directory` streams images directly from disk in batches — **no full-array RAM allocation**. This completely eliminates the MemoryError.

In [ ]:
# ── CNN generators (stream from disk — zero full-array RAM) ──────────────────
train_gen = datagen_train.flow_from_directory(
    dataset_path,
    target_size  = (IMAGE_SIZE, IMAGE_SIZE),
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    subset       = 'training',
    shuffle      = True,
    seed         = 42
)

val_gen = datagen_val.flow_from_directory(
    dataset_path,
    target_size  = (IMAGE_SIZE, IMAGE_SIZE),
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    subset       = 'validation',
    shuffle      = False,
    seed         = 42
)

num_classes = train_gen.num_classes
class_names = list(train_gen.class_indices.keys())

print(f"\n✅ Generators ready — images stream from disk, not RAM.")
print(f"   Training   batches : {len(train_gen)}  ({train_gen.n} images)")
print(f"   Validation batches : {len(val_gen)}  ({val_gen.n} images)")
print(f"   Classes            : {num_classes}")

## ✂️ Step 5 — Train / Test Split  `[Siri]`
The 80/20 split is handled by `validation_split=0.2` inside the generators. Here we create the `LabelEncoder` using the generator's class index for SVM/RF use.

In [ ]:
# Fit encoder on class names from the generator
encoder = LabelEncoder()
encoder.fit(class_names)

print(f"✅ LabelEncoder fitted on {len(encoder.classes_)} classes.")
print(f"   First 5: {list(encoder.classes_)[:5]} ...")
print()
print("   Train/Val split is already handled by flow_from_directory")
print(f"   Training images  : {train_gen.n}")
print(f"   Validation images: {val_gen.n}")

## ⚙️ Step 6 — Feature Processing (PCA for SVM / RF)  `[Ranjith]`
SVM and RF cannot scale to 60 K images, so we draw a **5 000-image sample** from disk using a separate generator, then apply PCA(100).

In [ ]:
SAMPLE_SIZE = 5000   # sufficient for SVM/RF, keeps RAM low

# Separate generator — no augmentation, no validation_split
datagen_flat = ImageDataGenerator(rescale=1./255)
flat_gen = datagen_flat.flow_from_directory(
    dataset_path,
    target_size = (IMAGE_SIZE, IMAGE_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'sparse',    # integer labels
    shuffle     = True,
    seed        = 42
)

# Collect SAMPLE_SIZE images
sample_x, sample_y = [], []
for imgs, lbls in flat_gen:
    sample_x.append(imgs)
    sample_y.append(lbls)
    collected = sum(len(b) for b in sample_x)
    if collected >= SAMPLE_SIZE:
        break

sample_x = np.concatenate(sample_x, axis=0)[:SAMPLE_SIZE]   # (5000, 32, 32, 3)
sample_y = np.concatenate(sample_y, axis=0)[:SAMPLE_SIZE].astype(int)

# Flatten → PCA
X_flat = sample_x.reshape(len(sample_x), -1).astype(np.float32)
del sample_x; gc.collect()

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_flat, sample_y,
    test_size  = 0.2,
    random_state = 42,
    stratify   = sample_y
)
del X_flat; gc.collect()

pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca  = pca.transform(X_test_s)
del X_train_s, X_test_s; gc.collect()

print(f"✅ Feature Processing complete.")
print(f"   Sample drawn       : {SAMPLE_SIZE} images")
print(f"   PCA input          : {X_train_pca.shape[1]+pca.n_components_} → 100 features")
print(f"   Train PCA shape    : {X_train_pca.shape}")
print(f"   Test  PCA shape    : {X_test_pca.shape}")
print(f"   Variance explained : {pca.explained_variance_ratio_.sum()*100:.1f}%")

## 🤖 Step 7 — Model Training  `[Vaseem]`
### 7a · SVM

In [ ]:
print("Training SVM on 5 000-sample PCA features ...")
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_pca, y_train_s)
print("✅ SVM training complete.")

### 7b · Random Forest

In [ ]:
print("Training Random Forest ...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_pca, y_train_s)
print("✅ Random Forest training complete.")

### 7c · CNN  *(streams from disk via generator — no MemoryError)*

In [ ]:
gc.collect()
tf.keras.backend.clear_session()

model = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', padding='same',
           input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    # Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    # Head
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer = 'adam',
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
model.summary()

In [ ]:
# Generator-based fit — only one batch (32 images) in RAM at a time
history = model.fit(
    train_gen,
    epochs          = 10,
    validation_data = val_gen,
    verbose         = 1
)
print("\n✅ CNN training complete.")

## 📊 Step 8 — Model Evaluation  `[Rudresh]`

In [ ]:
# ── SVM ──────────────────────────────────────────────────────────────────────
svm_pred     = svm_model.predict(X_test_pca)
svm_accuracy = accuracy_score(y_test_s, svm_pred)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_pred      = rf_model.predict(X_test_pca)
rf_accuracy  = accuracy_score(y_test_s, rf_pred)

# ── CNN — batch-wise to avoid OOM ────────────────────────────────────────────
cnn_loss, cnn_accuracy = model.evaluate(val_gen, verbose=0)

y_pred_cnn, y_true_cnn = [], []
val_gen.reset()
for imgs, lbls in val_gen:
    preds = model.predict(imgs, verbose=0)
    y_pred_cnn.extend(np.argmax(preds, axis=1))
    y_true_cnn.extend(np.argmax(lbls,  axis=1))
    if len(y_true_cnn) >= val_gen.n:
        break
y_pred_cnn = np.array(y_pred_cnn[:val_gen.n])
y_true_cnn = np.array(y_true_cnn[:val_gen.n])

print(f"SVM Accuracy           : {svm_accuracy*100:.2f}%  (on {SAMPLE_SIZE}-sample subset)")
print(f"Random Forest Accuracy : {rf_accuracy*100:.2f}%  (on {SAMPLE_SIZE}-sample subset)")
print(f"CNN Accuracy           : {cnn_accuracy*100:.2f}%  (on full validation set)")
print("\n✅ Model Evaluation complete.")

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

cm_data_list = [
    (confusion_matrix(y_test_s, svm_pred),  "SVM Confusion Matrix",           "Blues"),
    (confusion_matrix(y_test_s, rf_pred),   "Random Forest Confusion Matrix", "Greens"),
    (confusion_matrix(y_true_cnn, y_pred_cnn), "CNN Confusion Matrix",        "Oranges"),
]

for ax, (cm_data, title, cmap) in zip(axes, cm_data_list):
    sns.heatmap(cm_data, annot=False, fmt='d', cmap=cmap, ax=ax,
                xticklabels=False, yticklabels=False)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices — All Models", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Classification Report — CNN

In [ ]:
target_names = [c.replace('___', ' — ').replace('_', ' ') for c in class_names]
print(classification_report(y_true_cnn, y_pred_cnn, target_names=target_names))

## 🏗️ Step 9 — Model Architecture & Testing  `[Shankar]`
CNN training curves and three-model accuracy comparison.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('CNN Accuracy per Epoch')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('CNN Loss per Epoch')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True)

plt.suptitle("CNN Training Curves", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
model_names = ['SVM', 'Random Forest', 'CNN']
accuracies  = [svm_accuracy*100, rf_accuracy*100, cnn_accuracy*100]
colors      = ['#4C72B0', '#55A868', '#C44E52']

plt.figure(figsize=(8, 5))
bars = plt.bar(model_names, accuracies, color=colors, edgecolor='black', width=0.5)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.ylim(0, 110)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print("\n📊 Final Summary:")
for name, acc in zip(model_names, accuracies):
    print(f"  {name:<20}: {acc:.2f}%")
best = model_names[accuracies.index(max(accuracies))]
print(f"\n🏆 Best Model : {best}")

## 🌐 Step 10 — Frontend Deployment  `[Sneha]`
Save all models and helper objects. Provide prediction utility for Flask.

In [ ]:
# Save CNN
model.save('plant_disease_cnn.h5')

# Save SVM, RF, PCA, Encoder
with open('svm_model.pkl',   'wb') as f: pickle.dump(svm_model, f)
with open('rf_model.pkl',    'wb') as f: pickle.dump(rf_model,  f)
with open('pca_encoder.pkl', 'wb') as f:
    pickle.dump({'pca': pca, 'encoder': encoder}, f)

print("✅ Saved:")
print("   plant_disease_cnn.h5")
print("   svm_model.pkl  |  rf_model.pkl  |  pca_encoder.pkl")

In [ ]:
def predict_plant_disease(image_path: str, model_type: str = 'cnn') -> dict:
    """
    Predict plant disease from a leaf image.

    Parameters
    ----------
    image_path : str        path to a .jpg / .png leaf image
    model_type : str        'cnn' | 'svm' | 'rf'

    Returns
    -------
    dict  {'predicted_class': str, 'confidence': str}  (confidence only for CNN)
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE)).astype(np.float32) / 255.0

    if model_type == 'cnn':
        probs      = model.predict(img[np.newaxis, ...], verbose=0)[0]
        pred_idx   = int(np.argmax(probs))
        confidence = float(probs[pred_idx])
        label      = encoder.classes_[pred_idx]
        return {"predicted_class": label, "confidence": f"{confidence*100:.2f}%"}

    elif model_type in ('svm', 'rf'):
        flat     = img.flatten().reshape(1, -1)
        flat_pca = pca.transform(flat)
        clf      = svm_model if model_type == 'svm' else rf_model
        pred     = clf.predict(flat_pca)[0]
        return {"predicted_class": encoder.classes_[pred]}

    raise ValueError("model_type must be 'cnn', 'svm', or 'rf'")

print("✅ predict_plant_disease() ready.")
print("   predict_plant_disease('leaf.jpg', model_type='cnn')")